# 06 Full-city maps and building-stock check  (requires the Earth Engine map exports and raw data)

Predicts building height wall to wall for Porto Alegre, Sao Paulo and Recife with the 187-band map model of notebook 05,
compares it with a blurred LiDAR reference, and checks the building stock (volume) error.

**Inputs:** `outputs/final_model/*.joblib` (notebook 05), the 16 GeoTIFFs per city from `gee/export_map_features.js`
in `gee_exports/frantz_map_features/`, and the footprints and nDSMs in `data_raw/` (`data/SOURCES.md`).
**Outputs:** `outputs/maps/` (prediction rasters, metrics, figures, `stock_*.csv`).
The CNN inputs for the same windows come from `gee/export_map_cnn_inputs.js` (optional).

In [ ]:
# Setup: make `config.py` importable whether Jupyter was started in the repo root or in notebooks/
import sys, pathlib
_here = pathlib.Path.cwd().resolve()
for _p in (_here, _here / "notebooks", _here.parent / "notebooks"):
    if (_p / "config.py").exists():
        sys.path.insert(0, str(_p)); break
else:
    raise FileNotFoundError("config.py not found: start Jupyter from the repository root or from notebooks/")
import config
print("repo root:", config.ROOT)

## 1. Predict the maps

In [ ]:
import os
# =====================================================================
# CITY MAP CELL (CMP263): wall-to-wall prediction, LiDAR reference, comparison.
# Run after export_map_features_gee.js has finished (16 GeoTIFFs per city in MAP_FEATURES).
#
# Per city:
#  1. Reads the 16 layer GeoTIFFs (187 bands, -9999 = outside the municipality or no data),
#     checks they share one grid, stacks them in the order of selected_features.txt.
#  2. Predicts every valid pixel with the MAP VARIANT of the final model: mean of the HistGB
#     refit on the 187 bands (HistGB_map.joblib) and the final SVR (SVR.joblib). SVR is fed
#     its 1,872-column input with the 1,685 unselected columns empty: its imputer fills them
#     and its feature selection then drops them, so its prediction is exactly the one it would
#     make on the full feature vector. Negative values -> 0. Frantz et al. (2021, 4.3.3.2)
#     pruning: predictions below 2 m -> 0 (band 2; band 1 keeps the unpruned value).
#  3. Builds the LiDAR reference on the SAME 10 m grid with the label rule of the samples
#     (Frantz Eq. 2): area-weighted mean height of the building parts whose centroid lies
#     within 50 m of the pixel centre; 0 where no part is within 50 m. Parts as in the sampler
#     (exploded polygons, height_m > 0, area >= 10 m2). Also a candidate mask (a centroid within
#     5 m: the pixels the training labels come from) and a background mask (no centroid within
#     100 m: the rule of the background samples).
#  4. Pixel metrics on candidate pixels and on background pixels, all pixels and only pixels
#     at least 100 m from every training point of the city; a figure per city (prediction,
#     reference, difference) and GeoTIFFs for QGIS.
#
# NOT RUN ON REAL DATA by the author: tested on synthetic rasters only.
# =====================================================================
import glob, os, numpy as np, pandas as pd, geopandas as gpd, rasterio, joblib
from rasterio.merge import merge
from scipy.spatial import cKDTree
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

DATA = config.OUT

MAP_FEATURES = config.GEE_MAP_FEATURES          # folder of the GEE exports (gee/export_map_features.js)
FINAL = f"{DATA}/final_model"
TABLE = str(config.TABLE)
FOOTPRINTS = config.FOOTPRINTS
HEIGHT_FIELD = "height_m"
OUT = f"{DATA}/maps"
CITIES = {"porto_alegre": "EPSG:32722", "sao_paulo": "EPSG:32723", "recife": "EPSG:32725"}
NODATA = -9999.0
PRUNE_M = 2.0
MIN_PART_AREA_M2 = 10.0
CHUNK = 50_000
N_JOBS = -1
VMAX, DMAX = 60, 20                                     # colour limits: height (m), difference (+-m)
os.makedirs(OUT, exist_ok=True)

# ---- model, feature order ----
if "RFThreshold" not in globals():        # SVR.joblib contains this class; same code as the tuning cell
    from sklearn.base import BaseEstimator, TransformerMixin
    from sklearn.ensemble import RandomForestRegressor

    class RFThreshold(BaseEstimator, TransformerMixin):
        def __init__(self, frac=0.10, n_trees=500, mtry=0.01, thr=1e-3, min_keep=20, min_rows=500, seed=0, n_jobs=1):
            self.frac, self.n_trees, self.mtry, self.thr = frac, n_trees, mtry, thr
            self.min_keep, self.min_rows, self.seed, self.n_jobs = min_keep, min_rows, seed, n_jobs

        def fit(self, X, y):
            rng = np.random.default_rng(self.seed)
            idx = rng.choice(len(y), size=min(len(y), max(int(self.frac * len(y)), self.min_rows)), replace=False)
            rf = RandomForestRegressor(n_estimators=self.n_trees, max_features=self.mtry, n_jobs=self.n_jobs,
                                       random_state=self.seed).fit(X[idx], y[idx])
            keep = np.where(rf.feature_importances_ > self.thr)[0]
            if len(keep) < self.min_keep:
                keep = np.argsort(rf.feature_importances_)[::-1][:self.min_keep]
            self.keep_ = np.sort(keep)
            return self

        def transform(self, X):
            return X[:, self.keep_]

_sf = f"{FINAL}/selected_features.txt"
_sf = _sf if os.path.exists(_sf) else str(config.SELECTED_FEATURES)     # written by notebook 05; else the shipped copy
sel = [l.strip() for l in open(_sf) if l.strip()]
hgb_map, svr = joblib.load(f"{FINAL}/HistGB_map.joblib"), joblib.load(f"{FINAL}/SVR.joblib")
META_COLS = {"sample_id", "city", "height_m", "is_background", "x_m", "y_m", "split",
             "system:index", ".geo", "n_parts_50m", "slice_lo"}
feats = [c for c in config.read_table(nrows=0).columns if c not in META_COLS]   # same rule as load_table
col_of = {f: i for i, f in enumerate(feats)}
assert all(s in col_of for s in sel), "a selected feature is missing from the table header"
sel_cols = np.array([col_of[s] for s in sel])
layer_of = lambda n: n.split("_")[0]
layers = list(dict.fromkeys(layer_of(s) for s in sel))
tab = config.read_table(usecols=["city", "x_m", "y_m", "split"])


def read_layer(city, layer):
    files = sorted(glob.glob(f"{MAP_FEATURES}/{city}_map_{layer}.tif") +
                   glob.glob(f"{MAP_FEATURES}/{city}_map_{layer}-*.tif"))     # GEE splits big exports
    if not files:
        raise FileNotFoundError(f"no export for {city} / {layer} in {MAP_FEATURES}")
    srcs = [rasterio.open(f) for f in files]
    if len(srcs) == 1:
        arr, tf_ = srcs[0].read(), srcs[0].transform
    else:
        arr, tf_ = merge(srcs, nodata=NODATA)
    crs, desc = srcs[0].crs, list(srcs[0].descriptions)
    for s in srcs: s.close()
    names = [n for n in sel if layer_of(n) == layer]
    if len(arr) != len(names):
        raise ValueError(f"{city} {layer}: {len(arr)} bands, expected {len(names)}")
    if all(desc) and set(desc) == set(names) and desc != names:      # use the band names if GEE wrote them
        arr = arr[[desc.index(n) for n in names]]
    return names, arr.astype("float32"), tf_, crs


def predict_chunk(x_sel):
    full = np.full((len(x_sel), len(feats)), np.nan, dtype="float32")
    full[:, sel_cols] = x_sel
    p = (np.clip(hgb_map.predict(x_sel), 0, None) + np.clip(svr.predict(full), 0, None)) / 2
    return p.astype("float32")


def parts_of(city, crs):
    g = gpd.read_file(FOOTPRINTS(city))
    g = g[g.geometry.notna() & ~g.geometry.is_empty].explode(index_parts=False)
    g = g[g.geometry.geom_type == "Polygon"].to_crs(crs)
    z, a = g[HEIGHT_FIELD].astype(float).to_numpy(), g.geometry.area.to_numpy()
    ok = (z > 0) & np.isfinite(z) & (a >= MIN_PART_AREA_M2)
    c = g.geometry.centroid[ok]
    return c.x.to_numpy(), c.y.to_numpy(), a[ok], z[ok]


def reference(cx, cy, a, z, tf_, shape):
    """Exact Frantz Eq. 2 on the grid: for every part, add a*z and a to all pixels whose centre is within 50 m."""
    H, W = shape
    x0, y0, px = tf_.c, tf_.f, tf_.a
    col0, row0 = np.floor((cx - x0) / px).astype(int), np.floor((y0 - cy) / px).astype(int)
    num, den = np.zeros(shape), np.zeros(shape)
    near5, near100 = np.zeros(shape, bool), np.zeros(shape, bool)
    for dr in range(-11, 12):
        for dc in range(-11, 12):
            r, c = row0 + dr, col0 + dc
            inb = (r >= 0) & (r < H) & (c >= 0) & (c < W)
            d = np.hypot(x0 + (c + 0.5) * px - cx, y0 - (r + 0.5) * px - cy)
            m50, m5, m100 = inb & (d <= 50), inb & (d <= 5), inb & (d < 100)
            np.add.at(num, (r[m50], c[m50]), a[m50] * z[m50])
            np.add.at(den, (r[m50], c[m50]), a[m50])
            near5[r[m5], c[m5]] = True
            near100[r[m100], c[m100]] = True
    ref = np.where(den > 0, num / np.maximum(den, 1e-9), 0.0).astype("float32")
    return ref, near5, ~near100


def stats(p, r):
    e = p - r
    return dict(n=int(len(e)), RMSE=float(np.sqrt(np.mean(e ** 2))) if len(e) else np.nan,
                MAE=float(np.mean(np.abs(e))) if len(e) else np.nan, bias=float(np.mean(e)) if len(e) else np.nan)


REDO = []                                                # cities to recompute even if their outputs exist


def exports_ready(city):
    return all(glob.glob(f"{MAP_FEATURES}/{city}_map_{l}.tif") + glob.glob(f"{MAP_FEATURES}/{city}_map_{l}-*.tif")
               for l in layers)


for city, crs_exp in CITIES.items():
    if os.path.exists(f"{OUT}/{city}_metrics.csv") and city not in REDO:
        print(f"\n{city}: already mapped (outputs in {OUT}); put it in REDO to recompute"); continue
    if not exports_ready(city):
        missing = [l for l in layers if not (glob.glob(f"{MAP_FEATURES}/{city}_map_{l}.tif") +
                                             glob.glob(f"{MAP_FEATURES}/{city}_map_{l}-*.tif"))]
        print(f"\n{city}: exports not complete yet (missing layers: {', '.join(missing)}); skipped"); continue
    print(f"\n{city}")
    rows = []
    stack, tf0, crs0, shape0 = [], None, None, None
    for layer in layers:
        names, arr, tf_, crs = read_layer(city, layer)
        if tf0 is None:
            tf0, crs0, shape0 = tf_, crs, arr.shape[1:]
        elif tf_ != tf0 or arr.shape[1:] != shape0:
            raise ValueError(f"{city} {layer}: grid differs from the first layer ({tf_} vs {tf0})")
        stack.append(arr)
    if crs0.to_string() != crs_exp:
        print(f"  NOTE: rasters are in {crs0}, expected {crs_exp}")
    X = np.concatenate(stack)                                  # (187, H, W), in selected_features order
    del stack
    X[X == NODATA] = np.nan
    H, W = shape0
    X = X.reshape(len(sel), -1).T
    valid = ~np.all(np.isnan(X), axis=1)                       # outside the municipality = all bands empty
    print(f"  grid {W} x {H} px, {valid.sum():,} valid pixels ({valid.sum() / 1e4:.0f} km2)")
    idx = np.where(valid)[0]
    chunks = [idx[i:i + CHUNK] for i in range(0, len(idx), CHUNK)]
    preds = Parallel(n_jobs=N_JOBS, verbose=5)(delayed(predict_chunk)(X[ch]) for ch in chunks)
    pred = np.full(H * W, np.nan, "float32")
    pred[idx] = np.concatenate(preds)
    pred = pred.reshape(H, W)
    pruned = np.where(pred < PRUNE_M, 0, pred)
    del X

    cx, cy, a, z = parts_of(city, crs0)
    ref, cand, bg = reference(cx, cy, a, z, tf0, (H, W))
    vmask = valid.reshape(H, W)
    ref = np.where(vmask, ref, np.nan)
    tr = tab[(tab.city == city) & (tab.split == "train")]
    rr, cc = np.mgrid[0:H, 0:W]
    xs, ys = tf0.c + (cc + 0.5) * tf0.a, tf0.f - (rr + 0.5) * tf0.a
    dtr = cKDTree(tr[["x_m", "y_m"]].to_numpy()).query(np.c_[xs.ravel(), ys.ravel()], k=1)[0].reshape(H, W) \
        if len(tr) else np.full((H, W), np.inf)
    far = dtr >= 100

    prof = dict(driver="GTiff", height=H, width=W, crs=crs0, transform=tf0, dtype="float32", nodata=np.nan,
                compress="deflate", tiled=True)
    with rasterio.open(f"{OUT}/{city}_pred.tif", "w", count=2, **prof) as d:
        d.write(np.stack([pred, pruned])); d.set_band_description(1, "pred"); d.set_band_description(2, "pred_pruned_2m")
    with rasterio.open(f"{OUT}/{city}_ref.tif", "w", count=3, **prof) as d:
        d.write(np.stack([ref, np.where(vmask, cand, np.nan), np.where(vmask, bg, np.nan)]).astype("float32"))
        for k, nme in enumerate(["ref_50m_area_weighted", "candidate_5m", "background_100m"], 1):
            d.set_band_description(k, nme)

    for subset, m in [("candidate pixels", vmask & cand), ("candidate pixels, >= 100 m from training", vmask & cand & far),
                      ("background pixels", vmask & bg), ("all valid pixels", vmask)]:
        s = stats(pruned[m], ref[m])
        if "background" in subset:
            s["share predicted > 0 after pruning"] = float(np.mean(pruned[m] > 0)) if m.any() else np.nan
        rows.append(dict(city=city, subset=subset, **s))
    m = vmask & cand
    print(f"  candidate pixels: RMSE {rows[-4]['RMSE']:.2f} m, bias {rows[-4]['bias']:+.2f} m (n = {m.sum():,}); "
          f"mean height over the window: predicted {np.nanmean(pruned[vmask]):.2f} m, reference {np.nanmean(ref[vmask]):.2f} m")
    pd.DataFrame(rows).round(3).to_csv(f"{OUT}/{city}_metrics.csv", index=False)
    del pred, pruned, ref, cand, bg, dtr, rr, cc, xs, ys

# ---- summary over every city mapped so far (this run or earlier runs) ----
done = [c for c in CITIES if os.path.exists(f"{OUT}/{c}_metrics.csv")]
if not done:
    raise SystemExit("no city mapped yet")
res = pd.concat([pd.read_csv(f"{OUT}/{c}_metrics.csv") for c in done], ignore_index=True)
res.round(3).to_csv(f"{OUT}/map_metrics.csv", index=False)
panels = []
for c in done:
    with rasterio.open(f"{OUT}/{c}_pred.tif") as d: p_ = d.read(2)
    with rasterio.open(f"{OUT}/{c}_ref.tif") as d: r_ = d.read(1)
    panels.append((c, p_, r_))
print("\nMap vs LiDAR reference (pruned prediction, metres):")
print(res.round(2).to_string(index=False))

# ---- figure: one row per city: prediction, reference, difference ----
fig, axs = plt.subplots(len(panels), 3, figsize=(13, 4.2 * len(panels)), squeeze=False)
cm_h = plt.get_cmap("viridis").copy(); cm_h.set_bad("white")
cm_d = plt.get_cmap("RdBu_r").copy(); cm_d.set_bad("white")
for i, (city, p, r) in enumerate(panels):
    for j, (img, title) in enumerate([(p, "predicted (Sentinel-1/2)"), (r, "LiDAR reference (50 m rule)"),
                                      (p - r, "predicted minus reference")]):
        ax = axs[i, j]
        im = ax.imshow(img, cmap=cm_d if j == 2 else cm_h, interpolation="nearest",
                       **({"norm": TwoSlopeNorm(0, -DMAX, DMAX)} if j == 2 else {"vmin": 0, "vmax": VMAX}))
        ax.set_title(f"{city.replace('_', ' ').title()}: {title}", fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values(): s.set_visible(False)
        fig.colorbar(im, ax=ax, shrink=0.75, label="m")
fig.tight_layout()
fig.savefig(f"{OUT}/map_comparison.png", dpi=200)
plt.show()
print(f"\nSaved in {OUT}: <city>_pred.tif, <city>_ref.tif, map_metrics.csv, map_comparison.png")

## 2. Blurred nDSM reference and map comparison

In [ ]:
# =====================================================================
# nDSM REFERENCE CELL (CMP263): compare the city maps with the LiDAR nDSM aggregated to 10 m.
# Run after map_predict_cell.py, for every city it has mapped (reads <city>_pred.tif for the grid).
#
# The 10 m reference layers are the same three as the CNN patch targets (patch_dataset_cell.py),
# computed on the map grid tile by tile (no full-resolution raster is held in memory):
#   ndsm_mean    mean nDSM of all valid LiDAR pixels in the 10 m cell (includes trees)
#   bldg_height  mean nDSM of the LiDAR pixels inside building footprints (NaN where no building)
#   bldg_frac    share of the cell covered by footprints
# Cells without nDSM data are NaN (São Paulo's nDSM covers less than its footprints).
#
# Why two scales: the model predicts the mean building height within 50 m of a pixel (the Frantz
# label), not the height of the roof in that pixel, so its 10 m map is smoother than the nDSM.
# Frantz et al. (2021, Sec. 6.3) make the same point and suggest aggregating to coarser cells.
# Metrics are therefore given at 10 m (building cells, bldg_frac >= 0.5) and on 100 m blocks
# (footprint-area-weighted mean height of the block vs the prediction averaged over the same
# building cells, blocks with >= 10% building cover), plus the share of open cells (no building
# within 20 m) that get a height > 0 after the 2 m pruning.
#
# NOT RUN ON REAL DATA by the author: tested on synthetic rasters only.
# =====================================================================
import os, numpy as np, pandas as pd, geopandas as gpd, rasterio, matplotlib.pyplot as plt
from rasterio.features import rasterize
from rasterio.transform import from_origin
from rasterio.warp import reproject, Resampling, transform_bounds
from rasterio.windows import Window
from scipy.ndimage import maximum_filter, gaussian_filter
from scipy.signal import fftconvolve
from shapely.geometry import box
from matplotlib.colors import TwoSlopeNorm

DATA = config.OUT
MAPS = f"{DATA}/maps"
FOOTPRINTS = config.FOOTPRINTS
NDSM = config.NDSM
CITIES = ["porto_alegre", "sao_paulo", "recife"]
MIN_PART_AREA_M2 = 10.0
BLOCK_PX = 128                 # tile size on the 10 m grid
AGG = 10                       # 10 x 10 cells = 100 m blocks
VMAX, DMAX = 60, 20
DISC_M = 50                    # radius of the label-like blur (Frantz label: building parts within 50 m)
GAUSS_SIGMA_M = 25             # Gaussian alternative (FWHM about 59 m)
MIN_DISC_COVER = 0.5           # blurred reference only where >= 50% of the disc has nDSM
SHOW_FRAC = 0.25               # figure 3 and its metrics: cells at least 25% covered by footprints
NODATA = -9999.0


def ref_tile(src, fp, tf_tile, shape, crs):
    H, W = shape
    west, north = tf_tile.c, tf_tile.f
    east, south = west + W * tf_tile.a, north + H * tf_tile.e
    l, b, r, t = transform_bounds(crs, src.crs, west, south, east, north)
    w0 = src.window(l, b, r, t)
    win = Window(int(np.floor(w0.col_off)) - 1, int(np.floor(w0.row_off)) - 1,
                 int(np.ceil(w0.width)) + 3, int(np.ceil(w0.height)) + 3)
    nod = src.nodata if src.nodata is not None else NODATA
    arr = src.read(1, window=win, boundless=True, fill_value=nod).astype("float64")
    wt = src.window_transform(win)
    valid = np.isfinite(arr) & (arr != nod)
    arr = np.where(valid, np.maximum(arr, 0.0), 0.0)
    idx = fp.sindex.query(box(l, b, r, t))
    bld = (rasterize([(g, 1) for g in fp.geometry.iloc[idx]], out_shape=arr.shape, transform=wt, fill=0,
                     dtype="uint8").astype(bool) if len(idx) else np.zeros(arr.shape, bool))

    def avg(a):
        out = np.zeros(shape, "float32")
        reproject(a.astype("float32"), out, src_transform=wt, src_crs=src.crs, dst_transform=tf_tile, dst_crs=crs,
                  resampling=Resampling.average)
        return out

    with np.errstate(divide="ignore", invalid="ignore"):
        v, wb = avg(valid), avg(bld & valid)
        ndsm_mean = np.where(v > 0, avg(arr) / v, np.nan)
        bldg_h = np.where(wb > 0, avg(arr * (bld & valid)) / wb, np.nan)
        frac = np.where(v > 0, avg(bld), np.nan)
    return np.stack([ndsm_mean, bldg_h, frac]).astype("float32")


def blurred_reference(bldg_h, frac, covered, px):
    """Building-area-weighted mean height around each cell: sum(h * frac) / sum(frac) over a 50 m disc
    (the Frantz label with the footprint area spread over its pixels instead of sitting at the part centroid)
    and over a Gaussian. Ground is not averaged in; 0 where no building area falls in the kernel."""
    hf = np.where(covered, np.nan_to_num(bldg_h) * np.nan_to_num(frac), 0.0)
    f = np.where(covered, np.nan_to_num(frac), 0.0)
    cov = covered.astype(float)
    r = int(DISC_M // px)
    yy, xx = np.mgrid[-r:r + 1, -r:r + 1]
    disc = ((xx * px) ** 2 + (yy * px) ** 2 <= DISC_M ** 2).astype(float)
    conv = lambda a: fftconvolve(a, disc, mode="same")
    num, den, cv = conv(hf), conv(f), conv(cov) / disc.sum()
    with np.errstate(invalid="ignore", divide="ignore"):
        d = np.where(den > 1e-6, num / den, 0.0)
    d[(cv < MIN_DISC_COVER) | ~covered] = np.nan
    sg = GAUSS_SIGMA_M / px
    gn, gd = gaussian_filter(hf, sg), gaussian_filter(f, sg)
    gc = gaussian_filter(cov, sg)
    with np.errstate(invalid="ignore", divide="ignore"):
        g = np.where(gd > 1e-4, gn / gd, 0.0)
    g[(gc < MIN_DISC_COVER) | ~covered] = np.nan
    return d.astype("float32"), g.astype("float32")


def block_mean(a, w, k):
    """Weighted mean over k x k blocks (NaN-safe); returns block means and the summed weights."""
    H, W = a.shape
    Hc, Wc = H // k * k, W // k * k
    a, w = a[:Hc, :Wc], w[:Hc, :Wc]
    ok = np.isfinite(a) & np.isfinite(w) & (w > 0)
    num = np.where(ok, a * w, 0).reshape(Hc // k, k, Wc // k, k).sum((1, 3))
    den = np.where(ok, w, 0).reshape(Hc // k, k, Wc // k, k).sum((1, 3))
    with np.errstate(invalid="ignore", divide="ignore"):
        return num / den, den


def stats(p, r):
    ok = np.isfinite(p) & np.isfinite(r)
    p, r = p[ok], r[ok]
    e = p - r
    return dict(n=int(ok.sum()), RMSE=float(np.sqrt(np.mean(e ** 2))) if len(e) else np.nan,
                MAE=float(np.mean(np.abs(e))) if len(e) else np.nan, bias=float(e.mean()) if len(e) else np.nan,
                r=float(np.corrcoef(p, r)[0, 1]) if len(e) > 2 else np.nan)


rows, panels = [], []
for city in CITIES:
    pp = f"{MAPS}/{city}_pred.tif"
    if not os.path.exists(pp):
        print(f"{city}: no map yet ({pp}); skipped"); continue
    with rasterio.open(pp) as d:
        pred, tf0, crs0, (H, W) = d.read(2), d.transform, d.crs, d.shape        # band 2 = pruned at 2 m
    out_ref = f"{MAPS}/{city}_ndsm10.tif"
    if os.path.exists(out_ref):
        with rasterio.open(out_ref) as d:
            R = d.read()
        print(f"{city}: nDSM reference loaded from {out_ref}")
    else:
        print(f"{city}: aggregating the nDSM to the map grid ({W} x {H} px) ...")
        fp = gpd.read_file(FOOTPRINTS(city))
        fp = fp[fp.geometry.notna() & ~fp.geometry.is_empty]
        src = rasterio.open(NDSM(city))
        fp = fp.to_crs(src.crs)
        fp = fp[fp.geometry.area >= MIN_PART_AREA_M2]; fp.sindex
        R = np.full((3, H, W), np.nan, "float32")
        for r0 in range(0, H, BLOCK_PX):
            for c0 in range(0, W, BLOCK_PX):
                h, w = min(BLOCK_PX, H - r0), min(BLOCK_PX, W - c0)
                if np.all(np.isnan(pred[r0:r0 + h, c0:c0 + w])):
                    continue                                   # tile entirely outside the municipality
                tft = from_origin(tf0.c + c0 * tf0.a, tf0.f + r0 * tf0.e, tf0.a, -tf0.e)
                R[:, r0:r0 + h, c0:c0 + w] = ref_tile(src, fp, tft, (h, w), crs0)
        src.close()
        R[:, np.isnan(pred)] = np.nan
        with rasterio.open(out_ref, "w", driver="GTiff", height=H, width=W, count=3, dtype="float32", crs=crs0,
                           transform=tf0, nodata=np.nan, compress="deflate", tiled=True) as d:
            d.write(R)
            for k, n in enumerate(["ndsm_mean", "bldg_height", "bldg_frac"], 1): d.set_band_description(k, n)
    ndsm_mean, bldg_h, frac = R
    covered = np.isfinite(frac) & np.isfinite(pred)                 # inside municipality and nDSM
    bcell = covered & (frac >= 0.5)
    open_ = covered & (maximum_filter(np.nan_to_num(frac), size=5) == 0)   # no building within 20 m
    s10 = stats(pred[bcell], bldg_h[bcell])
    pb, wsum = block_mean(pred, np.where(bcell | (covered & (frac > 0)), frac, np.nan), AGG)
    rb, _ = block_mean(bldg_h, frac, AGG)
    cov_b, _ = block_mean(np.where(covered, frac, np.nan), np.where(covered, 1.0, np.nan), AGG)
    okb = np.isfinite(pb) & np.isfinite(rb) & (cov_b >= 0.10)
    s100 = stats(pb[okb], rb[okb])
    disc_ref, gauss_ref = blurred_reference(bldg_h, frac, covered, tf0.a)
    with rasterio.open(f"{MAPS}/{city}_ndsm10_blurred.tif", "w", driver="GTiff", height=H, width=W, count=2,
                       dtype="float32", crs=crs0, transform=tf0, nodata=np.nan, compress="deflate", tiled=True) as d:
        d.write(np.stack([disc_ref, gauss_ref]))
        d.set_band_description(1, f"bldg_height_disc{DISC_M}m_area_weighted")
        d.set_band_description(2, f"bldg_height_gauss_sigma{GAUSS_SIGMA_M}m_area_weighted")
    near = np.isfinite(disc_ref) & (disc_ref > 0)                    # some building area within 50 m
    sd, sg_ = stats(pred[near], disc_ref[near]), stats(pred[near], gauss_ref[near])
    rows.append(dict(city=city, scale=f"10 m, vs {DISC_M} m disc-blurred nDSM (cells with buildings within {DISC_M} m)", **sd))
    rows.append(dict(city=city, scale=f"10 m, vs Gaussian-blurred nDSM (sigma {GAUSS_SIGMA_M} m), same cells", **sg_))
    bm = covered & (np.nan_to_num(frac) >= SHOW_FRAC) & np.isfinite(disc_ref)
    rows.append(dict(city=city, scale=f"10 m building cells (bldg_frac >= {SHOW_FRAC}), vs {DISC_M} m disc-blurred nDSM",
                     **stats(pred[bm], disc_ref[bm])))
    rows.append(dict(city=city, scale=f"10 m building cells (bldg_frac >= {SHOW_FRAC}), vs Gaussian-blurred nDSM",
                     **stats(pred[bm], gauss_ref[bm])))
    rows.append(dict(city=city, scale="10 m, building cells (bldg_frac >= 0.5)", **s10))
    rows.append(dict(city=city, scale="100 m blocks, >= 10% building cover", **s100))
    rows.append(dict(city=city, scale="open cells (no building within 20 m)", n=int(open_.sum()),
                     share_predicted_above_0=float(np.mean(pred[open_] > 0)) if open_.any() else np.nan,
                     mean_prediction=float(np.mean(pred[open_])) if open_.any() else np.nan))
    print(f"  nDSM coverage of the mapped window: {covered.sum() / np.isfinite(pred).sum():.0%}; "
          f"10 m building cells RMSE {s10['RMSE']:.2f} m, bias {s10['bias']:+.2f} m; "
          f"100 m blocks RMSE {s100['RMSE']:.2f} m, bias {s100['bias']:+.2f} m, r {s100['r']:.2f}; "
          f"vs {DISC_M} m disc-blurred nDSM RMSE {sd['RMSE']:.2f} m, bias {sd['bias']:+.2f} m, r {sd['r']:.2f}")
    panels.append((city, pred, np.where(covered, np.where(frac > 0, bldg_h, 0), np.nan), pb, rb, okb, disc_ref, gauss_ref,
                   covered & (np.nan_to_num(frac) >= SHOW_FRAC)))

res = pd.DataFrame(rows)
res.round(3).to_csv(f"{MAPS}/ndsm_comparison_metrics.csv", index=False)
print("\nPrediction (pruned at 2 m) vs LiDAR nDSM aggregated to 10 m:")
print(res.round(2).to_string(index=False))

# ---- figure 1: per city: prediction 10 m | nDSM building height 10 m | difference on 100 m blocks ----
cm_h = plt.get_cmap("viridis").copy(); cm_h.set_bad((0, 0, 0, 0))
cm_d = plt.get_cmap("RdBu_r").copy(); cm_d.set_bad((0, 0, 0, 0))
fig, axs = plt.subplots(len(panels), 3, figsize=(13, 4.3 * len(panels)), squeeze=False)
for i, (city, p, r10, pb, rb, okb, dref, gref, bm) in enumerate(panels):
    diff = np.where(okb, pb - rb, np.nan)
    for j, (img, title, cm, kw) in enumerate([
            (p, "predicted, 10 m (Sentinel-1/2)", cm_h, dict(vmin=0, vmax=VMAX)),
            (r10, "LiDAR nDSM, building height, 10 m", cm_h, dict(vmin=0, vmax=VMAX)),
            (diff, "predicted minus LiDAR, 100 m blocks", cm_d, dict(norm=TwoSlopeNorm(0, -DMAX, DMAX)))]):
        ax = axs[i, j]
        im = ax.imshow(img, cmap=cm, interpolation="nearest", **kw)
        ax.set_title(f"{city.replace('_', ' ').title()}: {title}", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values(): s.set_visible(False)
        fig.colorbar(im, ax=ax, shrink=0.75, label="m")
fig.tight_layout(); fig.savefig(f"{MAPS}/map_vs_ndsm.png", dpi=200); plt.show()

# ---- figure 2: 100 m blocks, predicted vs LiDAR ----
fig, axs = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 4), squeeze=False)
for ax, (city, p, r10, pb, rb, okb, dref, gref, bm) in zip(axs[0], panels):
    ax.hexbin(rb[okb], pb[okb], gridsize=35, bins="log", cmap="Greys", mincnt=1, extent=(0, VMAX, 0, VMAX))
    ax.plot([0, VMAX], [0, VMAX], color="#2a6fdb", lw=1)
    s = res[(res.city == city) & res.scale.str.startswith("100 m")].iloc[0]
    ax.set_title(f"{city.replace('_', ' ').title()}: RMSE {s.RMSE:.1f} m, bias {s.bias:+.1f} m, r {s.r:.2f}", fontsize=9)
    ax.set_xlabel("LiDAR building height, 100 m block (m)"); ax.set_ylabel("predicted (m)")
    ax.set_xlim(0, VMAX); ax.set_ylim(0, VMAX); ax.set_aspect("equal")
    for s_ in ["top", "right"]: ax.spines[s_].set_visible(False)
fig.tight_layout(); fig.savefig(f"{MAPS}/blocks_100m_scatter.png", dpi=200); plt.show()
# ---- figure 3: same blur level, shown on building cells only (cells >= SHOW_FRAC footprint cover) ----
# The blurred references average buildings only (like the Frantz label), so around an isolated building they form a
# plateau with a sharp edge; masking both maps to the building cells removes those edges and compares like with like.
fig, axs = plt.subplots(len(panels), 5, figsize=(21, 4.0 * len(panels)), squeeze=False)
for i, (city, p, r10, pb, rb, okb, dref, gref, bm) in enumerate(panels):
    M = lambda a: np.where(bm, a, np.nan)
    for j, (img, title, cm, kw) in enumerate([
            (M(p), "predicted, 10 m", cm_h, dict(vmin=0, vmax=VMAX)),
            (M(r10), "LiDAR nDSM, 10 m", cm_h, dict(vmin=0, vmax=VMAX)),
            (M(dref), f"LiDAR, {DISC_M} m disc, area-weighted", cm_h, dict(vmin=0, vmax=VMAX)),
            (M(gref), f"LiDAR, Gaussian sigma {GAUSS_SIGMA_M} m", cm_h, dict(vmin=0, vmax=VMAX)),
            (M(p - gref), "predicted minus Gaussian", cm_d, dict(norm=TwoSlopeNorm(0, -DMAX, DMAX)))]):
        ax = axs[i, j]
        ax.set_facecolor("#e6e6e6")
        im = ax.imshow(img, cmap=cm, interpolation="nearest", **kw)
        ax.set_title(f"{city.replace('_', ' ').title()}: {title}", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
        for s_ in ax.spines.values(): s_.set_visible(False)
        fig.colorbar(im, ax=ax, shrink=0.7, label="m")
fig.tight_layout(); fig.savefig(f"{MAPS}/map_vs_ndsm_blurred.png", dpi=200); plt.show()
print(f"\nSaved in {MAPS}: <city>_ndsm10.tif, <city>_ndsm10_blurred.tif, map_vs_ndsm_blurred.png, ndsm_comparison_metrics.csv, map_vs_ndsm.png, blocks_100m_scatter.png")

## 3. Building stock: do the over- and underestimations cancel?

In [ ]:
# =====================================================================
# BUILDING STOCK CELL (CMP263): do the over- and underestimations of the city maps cancel out?
# Run after ndsm_reference_cell.py (reads <city>_pred.tif and <city>_ndsm10.tif from MAPS).
#
# Quantity compared: building VOLUME, the proxy for building stock.
#   reference volume of a 10 m cell  V_ref  = bldg_frac * bldg_height * 100 m2   (LiDAR nDSM inside footprints)
#   predicted volume of the same cell V_pred = bldg_frac * prediction  * 100 m2
# The model predicts height only, not footprints, so the predicted volume borrows the reference
# footprint area (bldg_frac). This isolates the HEIGHT error: it is the volume you would get by
# combining our height map with a perfect footprint layer. A real application would use OSM, Google
# Open Buildings or a classified footprint map instead, which adds the footprint error on top.
#
# Error indicators, per city and per aggregation scale (10 m cells, 100 m, 500 m, 1 km blocks, whole window):
#   total signed error   sum(V_pred - V_ref), and as % of the reference stock
#   sum absolute error   sum(|V_pred - V_ref|) over the units of that scale, and as % of the stock (WAPE)
#   over / under         sum of the positive and of the negative unit errors
#   cancellation ratio   |total signed error| / sum absolute error: 1 = all errors go the same way,
#                        0 = they cancel completely. At the whole-window scale the two sums coincide.
# The sum absolute error shrinks as the units grow, because errors cancel inside each unit; the
# total signed error is the same at every scale. Comparing the two answers the question directly.
# Also: the signed volume error by reference height class (where the over- and underestimation
# come from) and a 1 km block bootstrap CI of the total signed error (blocks resampled, 1,000 times).
#
# Cells used: inside the municipality window, nDSM available, bldg_frac > 0, bldg_height finite.
# São Paulo's nDSM covers less ground than its footprints: those cells are left out of BOTH sums.
# Note: the training labels come from the providers' per-part height statistic (median part height
# in these three cities), while V_ref uses the mean nDSM inside the footprints, so part of a total
# bias can be a label-definition offset, not a model error.
#
# NOT RUN ON REAL DATA by the author: tested on synthetic rasters only.
# =====================================================================
import os, numpy as np, pandas as pd, rasterio

DATA = config.OUT
MAPS = globals().get("MAPS", f"{DATA}/maps")
CITIES = ["porto_alegre", "sao_paulo", "recife"]
SCALES_PX = {"10 m cells": 1, "100 m blocks": 10, "500 m blocks": 50, "1 km blocks": 100}
CLASS_EDGES = [2, 6, 15, 30]
CLASS_NAMES = ["<2 m", "2-6 m", "6-15 m", "15-30 m", ">30 m"]
N_BOOT, BOOT_PX, SEED = 1000, 100, 42
VARIANTS = {"pruned at 2 m": 2, "raw": 1}            # band of <city>_pred.tif


def block_sum(a, k):
    """Sum over k x k blocks (NaN-free input); the ragged edge is padded with zeros."""
    if k == 1:
        return a
    H, W = a.shape
    Hp, Wp = -(-H // k) * k, -(-W // k) * k
    b = np.zeros((Hp, Wp), a.dtype); b[:H, :W] = a
    return b.reshape(Hp // k, k, Wp // k, k).sum(axis=(1, 3))


rows, cls_rows, totals = [], [], []
for city in CITIES:
    pp, rp = f"{MAPS}/{city}_pred.tif", f"{MAPS}/{city}_ndsm10.tif"
    if not (os.path.exists(pp) and os.path.exists(rp)):
        print(f"{city}: {pp if not os.path.exists(pp) else rp} missing; run map_predict_cell / ndsm_reference_cell first")
        continue
    with rasterio.open(rp) as d:
        _, bldg_h, frac = d.read().astype("float64")
        cell_m2 = abs(d.transform.a * d.transform.e)
    for variant, band in VARIANTS.items():
        with rasterio.open(pp) as d:
            pred = d.read(band).astype("float64")
        use = np.isfinite(pred) & np.isfinite(frac) & (frac > 0) & np.isfinite(bldg_h)
        no_ndsm = np.isfinite(pred) & np.isfinite(frac) & (frac > 0) & ~np.isfinite(bldg_h)
        fa = np.where(use, frac * cell_m2, 0.0)                     # footprint area per cell, m2
        v_ref = fa * np.where(use, bldg_h, 0.0)
        v_pred = fa * np.where(use, np.clip(pred, 0, None), 0.0)
        dv = v_pred - v_ref
        stock, signed = v_ref.sum(), dv.sum()
        for scale, k in list(SCALES_PX.items()) + [("whole window", None)]:
            e = np.array([signed]) if k is None else block_sum(dv, k)[block_sum(fa, k) > 0]
            sae = np.abs(e).sum()
            rows.append(dict(city=city, prediction=variant, scale=scale, units=len(e),
                             stock_ref_Mm3=stock / 1e6, stock_pred_Mm3=v_pred.sum() / 1e6,
                             total_signed_error_Mm3=signed / 1e6, total_signed_error_pct=100 * signed / stock,
                             sum_abs_error_Mm3=sae / 1e6, sum_abs_error_pct=100 * sae / stock,
                             overestimation_Mm3=e[e > 0].sum() / 1e6, underestimation_Mm3=e[e < 0].sum() / 1e6,
                             cancellation_ratio=abs(signed) / sae if sae > 0 else np.nan))
        # where the errors come from: signed volume error by reference height class (10 m cells)
        c = np.digitize(bldg_h, CLASS_EDGES)
        for j, nm in enumerate(CLASS_NAMES):
            m = use & (c == j)
            cls_rows.append(dict(city=city, prediction=variant, ref_class=nm,
                                 share_of_stock_pct=100 * v_ref[m].sum() / stock,
                                 signed_error_Mm3=dv[m].sum() / 1e6, signed_error_pct_of_stock=100 * dv[m].sum() / stock,
                                 abs_error_pct_of_stock=100 * np.abs(dv[m]).sum() / stock))
        # 1 km block bootstrap of the total signed error (% of stock)
        bd, br = block_sum(dv, BOOT_PX).ravel(), block_sum(v_ref, BOOT_PX).ravel()
        keep = block_sum(fa, BOOT_PX).ravel() > 0
        bd, br = bd[keep], br[keep]
        rng = np.random.default_rng(SEED)
        idx = rng.integers(0, len(bd), size=(N_BOOT, len(bd)))
        boot = 100 * bd[idx].sum(1) / br[idx].sum(1)
        totals.append(dict(city=city, prediction=variant, stock_ref_Mm3=stock / 1e6, stock_pred_Mm3=v_pred.sum() / 1e6,
                           total_signed_error_pct=100 * signed / stock,
                           ci95_low=np.percentile(boot, 2.5), ci95_high=np.percentile(boot, 97.5),
                           footprint_area_km2=fa.sum() / 1e6,
                           ref_mean_height_m=stock / fa.sum(), pred_mean_height_m=v_pred.sum() / fa.sum(),
                           building_cells=int(use.sum()), building_cells_without_ndsm=int(no_ndsm.sum())))

res, cls, tot = pd.DataFrame(rows), pd.DataFrame(cls_rows), pd.DataFrame(totals)
res.round(4).to_csv(f"{MAPS}/stock_error_by_scale.csv", index=False)
cls.round(4).to_csv(f"{MAPS}/stock_error_by_class.csv", index=False)
tot.round(4).to_csv(f"{MAPS}/stock_totals.csv", index=False)
pd.set_option("display.width", 220)
main = "pruned at 2 m"
print("1. Total building volume (reference footprints x height), map vs LiDAR. CI: 1 km block bootstrap")
print(tot.round(2).to_string(index=False))
print(f"\n2. Sum absolute error vs total signed error by scale ({main}); % of the reference stock")
print(res[res.prediction == main][["city", "scale", "units", "total_signed_error_pct", "sum_abs_error_pct",
                                   "overestimation_Mm3", "underestimation_Mm3", "cancellation_ratio"]]
      .round(3).to_string(index=False))
print(f"\n3. Signed volume error by reference height class (10 m cells, {main}), % of the city stock")
print(cls[cls.prediction == main].pivot(index="ref_class", columns="city", values="signed_error_pct_of_stock")
      .reindex(CLASS_NAMES).round(2).to_string())
print(f"\nSaved in {MAPS}: stock_totals.csv, stock_error_by_scale.csv, stock_error_by_class.csv")